### Links and Resources
| Concept / Command | Description |
|-------------------|-------------|
| [`pyspark.sql.functions`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html) | Master list of all built-in column functions |
| [Spark SQL function index](https://spark.apache.org/docs/latest/api/sql/index.html) | Same functions, SQL syntax |
| [Catalyst optimizer](https://databricks.com/glossary/catalyst-optimizer) | Why built-ins beat UDFs |
| [Photon engine](https://www.databricks.com/product/photon) | Vectorized C++ runtime — only accelerates built-ins |

In [ ]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark, display  # local run through Databricks Connect

from pyspark.sql import functions as F

# Introduction to Functions

Almost every transformation you write in Spark goes through a **function** — `concat`, `when`, `to_date`, `sum`, `row_number`. They live in two places:

- **Python:** `pyspark.sql.functions` — by convention imported as `F`.
- **SQL:** the same names, called inline in `spark.sql(...)` or `selectExpr(...)`.

This notebook is the map. The next six drill into specific categories.

## Why this matters: built-in vs. UDF

A *built-in* function is recognized by Catalyst (Spark's optimizer) and Photon (the vectorized engine). It can be:

- Pushed into the data source (predicate / projection pushdown).
- Reordered, fused with neighbours, or eliminated entirely.
- Executed in native code without crossing the JVM↔Python boundary.

A Python UDF can do *none* of those things. Same logic, 5–50× slower. **Always reach for a built-in first.**

In [ ]:
# The single most useful import in PySpark
from pyspark.sql import functions as F

# F now exposes ~400 functions. A few you will use every day:
print([f for f in dir(F) if f.startswith("to_")][:10])
print([f for f in dir(F) if "date" in f.lower()][:10])

## Three ways to call the same function

PySpark gives you parallel APIs. They compile to the same plan — pick whichever reads best.

In [ ]:
df = spark.range(5).withColumnRenamed("id", "n")

# 1. DataFrame API
df.select(F.col("n"), F.col("n") * 2).show(3)

# 2. selectExpr — SQL fragments
df.selectExpr("n", "n * 2 AS doubled").show(3)

# 3. spark.sql against a temp view
df.createOrReplaceTempView("nums")
spark.sql("SELECT n, n * 2 AS doubled FROM nums LIMIT 3").show()

## Function categories at a glance

| Category | Examples | Notebook |
|---|---|---|
| String | `concat`, `regexp_replace`, `substring`, `split` | 02 |
| Numeric / math | `round`, `abs`, `pow`, `sqrt`, `rand` | 03 |
| Date / timestamp | `current_date`, `date_add`, `to_date`, `date_format` | 04 |
| Conditional | `when/otherwise`, `coalesce`, `nullif` | 05 |
| Collection (arrays) | `array_contains`, `explode`, `size`, `flatten` | 06 |
| Map / struct / JSON | `map_keys`, `from_json`, `to_json`, `struct` | 07 |
| Higher-order | `transform`, `filter`, `aggregate` (lambdas) | 08 |
| Window | `row_number`, `lag`, `rank` over partitions | 09 |
| Aggregation | `sum`, `count_distinct`, `percentile_approx` | 10 |

## Reading function signatures

When you don't remember a function's arguments, the cheapest lookup is `help()` in the notebook itself:

In [ ]:
help(F.regexp_replace)

## A tiny demo dataset

We'll reuse this shape across notebooks 02–07. Make sure it loads.

In [ ]:
people = spark.createDataFrame([
    (1, "Alice",   "alice@example.com",  "2024-03-15", 1234.50),
    (2, "Bob",     "BOB@EXAMPLE.COM",    "2024-04-02",  987.00),
    (3, "Charlie", None,                  "2023-12-31", 4500.99),
    (4, "Dana",    "dana@example.com",   "2024-06-10",    None),
    (5, "Erin",    " erin@example.com ", "2025-01-20",  150.25),
], ["id", "name", "email", "signup_date", "balance"])

display(people)

## Takeaways

- `from pyspark.sql import functions as F` is the standard import.
- DataFrame API, `selectExpr`, and `spark.sql` are interchangeable — pick what reads best.
- Always prefer a built-in to a UDF: Catalyst can optimize built-ins, Photon can vectorize them.
- The function namespace is huge but well-organized by category — notebooks 02–10 walk through them.